# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiezugwu/Data-Science/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np

D = pd.read_parquet("decline_recovery.parquet")
R = pd.read_parquet("ranking_lifecycle.parquet")[["content_hash_id", "search_volume"]]
D = D.merge(R, on="content_hash_id", how="left")

# derived fields
D["first30"] = D.impressions_90d - D.impressions_last_30d - D.impressions_prev_30d
D["severity"] = -(D.impressions_last_30d / D.impressions_prev_30d - 1) * 100
D["prev30_ctr_pct"] = D.clicks_prev_30d / D.impressions_prev_30d * 100

fields = ["severity", "impressions_prev_30d", "first30", "prev30_ctr_pct",
          "content_age_days", "search_volume"]

# 1. summary stats per field
stats = pd.DataFrame({
    "n":      D[fields].notna().sum(),
    "nulls":  D[fields].isna().sum(),
    "zeros":  (D[fields] == 0).sum(),
    "median": D[fields].median(),
    "mean":   D[fields].mean(),
    "p99":    D[fields].quantile(0.99),
    "max":    D[fields].max(),
}).round(2)

# 2. tail concentration: share of the total held by the top 1% and top 10% of pages
def top_share(s, frac):
    s = s.sort_values(ascending=False)
    return s.head(int(len(s) * frac)).sum() / s.sum()

# 3. histogram on a log scale (14 equal-width bins)
counts, edges = np.histogram(np.log10(D.impressions_prev_30d + 1), bins=14)

# 4. CTR histogram, clipped at the 99.5th percentile for readability
ctr_clipped = D.prev30_ctr_pct.clip(upper=D.prev30_ctr_pct.quantile(0.995))
counts_ctr, edges_ctr = np.histogram(ctr_clipped, bins=14)
print(stats)
print("top 1% share:", round(top_share(D.impressions_prev_30d, 0.01), 3))
print("top 10% share:", round(top_share(D.impressions_prev_30d, 0.10), 3))
print("prior-30 impressions, 14 log bins:", counts.tolist())
print("CTR, 14 bins:", np.histogram(ctr_clipped, bins=14)[0].tolist())

                          n  nulls  zeros  median     mean       p99  \
severity              56253      0      0   57.85    60.24    100.00   
impressions_prev_30d  56253      0      0  536.00  2091.04  22755.16   
first30               56253      0   3746  674.00  2659.64  28727.20   
prev30_ctr_pct        56253      0  24156    0.10     0.26      1.87   
content_age_days      56253      0      0  193.00   230.15    550.00   
search_volume         54520   1733  24080   10.00   118.45   1600.00   

                            max  
severity                 100.00  
impressions_prev_30d  239803.00  
first30               259966.00  
prev30_ctr_pct             9.47  
content_age_days         577.00  
search_volume         201000.00  
top 1% share: 0.2
top 10% share: 0.619
prior-30 impressions, 14 log bins: [9944, 9338, 8787, 7623, 6273, 4932, 3564, 2443, 2218, 690, 301, 100, 28, 12]
CTR, 14 bins: [32349, 8155, 5855, 3334, 2214, 1466, 775, 587, 401, 267, 222, 167, 101, 360]


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.